# CQD-SHAP Colab: 2000 Query Demo

This notebook runs a smaller CQD-SHAP experiment for report/demo use. It limits each selected query type to `MAX_QUERIES = 2000`, so it is faster than the full benchmark while still covering more queries than the quick demo.

Important: results from 2000 queries are subset/demo results, not full paper reproduction results.

## 1. Runtime Check

Use `Runtime -> Change runtime type -> T4 GPU` before running.

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone Project

In [ ]:
%cd /content
!rm -rf CQD-SHAP
!git clone https://github.com/ds-jrg/CQD-SHAP.git
%cd /content/CQD-SHAP
!pip -q install gdown pandas networkx matplotlib tqdm

## 3. Download Data and Models

In [ ]:
%cd /content/CQD-SHAP

!test -f data.zip || gdown 1yoZFUAY7DLOj4fC78pIU32SUSAEWRmLw -O data.zip
!test -d data || unzip -q data.zip

!test -f models.zip || gdown 1ot3CuVk4DorVu3JiHKzdumzGNaTREAU3 -O models.zip
!test -d models || unzip -q models.zip

## 4. Check Available Files

In [ ]:
from pathlib import Path

paths = {
    "evaluation.py": "evaluation.py",
    "FB15k-237 data": "data/FB15k-237",
    "FB15k-237+H data": "data/FB15k-237+H",
    "Freebase model": "models/FB15k-237-model-rank-1000-epoch-100-1602508358.pt",
}

for label, path in paths.items():
    print(f"{label}: {Path(path).exists()} -> {path}")

assert Path("evaluation.py").exists(), "Missing evaluation.py"
assert Path("data/FB15k-237").exists(), "Missing data/FB15k-237"
assert Path("models/FB15k-237-model-rank-1000-epoch-100-1602508358.pt").exists(), "Missing Freebase model"

## 5. Create Limited Evaluator

This creates `evaluation_limited.py` from `evaluation.py` and adds `--max_queries`.

In [ ]:
from pathlib import Path

src_path = Path("evaluation.py")
limited_path = Path("evaluation_limited.py")
src = src_path.read_text()

if "--max_queries" not in src:
    src = src.replace(
        "parser.add_argument('--normalize', action='store_true', help='Whether to normalize scores using sigmoid')",
        "parser.add_argument('--normalize', action='store_true', help='Whether to normalize scores using sigmoid')\n"
        "    parser.add_argument('--max_queries', type=int, help='Limit number of queries per query type')"
    )
    src = src.replace(
        "        hard = query_dataset_hard.get_queries(query_type)\n"
        "        complete = query_dataset.get_queries(query_type)\n",
        "        hard = query_dataset_hard.get_queries(query_type)\n"
        "        complete = query_dataset.get_queries(query_type)\n"
        "        if args.max_queries is not None:\n"
        "            hard = hard[:args.max_queries]\n"
        "            complete = complete[:args.max_queries]\n"
        "            logging.info(f'Limited {query_type} to {len(hard)} queries')\n"
    )

limited_path.write_text(src)
print("Created", limited_path)
!python evaluation_limited.py --help | grep -E "max_queries|method|query_type"

## 6. Choose Demo Size

Default: up to 2000 queries for query type `2p`. If a query type has fewer than 2000 queries, it will run all available queries for that type.

In [ ]:
from pathlib import Path

MAX_QUERIES = 2000
QUERY_TYPES = ["2p"]
METHODS = ["shapley", "score", "random", "first", "last"]

if Path("data/FB15k-237+H").exists():
    benchmark_version = 2
    data_dir = "data/FB15k-237+H"
else:
    benchmark_version = 1
    data_dir = "data/FB15k-237"

model_path = "models/FB15k-237-model-rank-1000-epoch-100-1602508358.pt"

print("MAX_QUERIES:", MAX_QUERIES)
print("QUERY_TYPES:", QUERY_TYPES)
print("METHODS:", METHODS)
print("benchmark_version:", benchmark_version)
print("data_dir:", data_dir)
print("model_path:", model_path)

## 7. Run 2000-Query Experiment

This skips a run if both CSV outputs already exist.

In [ ]:
import subprocess
from pathlib import Path

output_dir = Path(f"evaluation_benchmark{benchmark_version}/Freebase")

for query_type in QUERY_TYPES:
    for method in METHODS:
        necessary_file = output_dir / f"bench{benchmark_version}_{query_type}_{method}_necessary.csv"
        sufficient_file = output_dir / f"bench{benchmark_version}_{query_type}_{method}_sufficient.csv"

        if necessary_file.exists() and sufficient_file.exists():
            print(f"SKIP query_type={query_type}, method={method}: outputs already exist")
            continue

        print(f"RUN query_type={query_type}, method={method}, max_queries={MAX_QUERIES}")
        subprocess.run([
            "python", "evaluation_limited.py",
            "--kg", "Freebase",
            "--benchmark", str(benchmark_version),
            "--query_type", query_type,
            "--method", method,
            "--data_dir", data_dir,
            "--model_path", model_path,
            "--max_queries", str(MAX_QUERIES),
        ], check=True)

## 8. Summarize Outputs

In [ ]:
from pathlib import Path
import pandas as pd

files = sorted(Path(".").glob("evaluation_benchmark*/**/*.csv"))
print("CSV files found:", len(files))
for path in files:
    print("-", path)

rows = []
for path in files:
    df = pd.read_csv(path)
    rows.append({
        "file": str(path),
        "rows/question_answer_cases": len(df),
        "unique_queries": df["query_idx"].nunique() if "query_idx" in df.columns else None,
        "query_types": ",".join(sorted(df["query_type"].unique())) if "query_type" in df.columns else "",
        "mean_delta_mrr": df["delta_mrr"].mean() if "delta_mrr" in df.columns else None,
        "mean_delta_hit_1": df["delta_hit_1"].mean() if "delta_hit_1" in df.columns else None,
        "mean_runtime": df["runtime"].mean() if "runtime" in df.columns else None,
    })

summary = pd.DataFrame(rows)
summary

## 9. Save Summary CSV

In [ ]:
summary_path = Path("evaluation_summary_2000_queries.csv")
summary.to_csv(summary_path, index=False)
print("Saved", summary_path)
!ls -lh evaluation_summary_2000_queries.csv